# Intro to Google Cloud's Gemini API (Healthcare Migration)

## Overview

This notebook demonstrates how to utilize Google Cloud's latest generative AI models via Vertex AI. This version has been updated to use the active Gemini models, succeeding the deprecated MedLM endpoints.

## Learning objectives

Learn how to utilize Gemini via Python and the REST API.

## Get started

### Install the Modern SDK

In [ ]:
!pip install google-genai

### Gemini with Python SDK

To invoke the model via Python, we initialize the modern Google GenAI client. We will also dynamically fetch your Cloud Lab environment settings.

In [ ]:
import subprocess
from google import genai
from google.genai import types

# Dynamically fetch the current active Cloud Lab GCP project ID
PROJECT_ID = subprocess.check_output("gcloud config get-value project", shell=True).decode("utf-8").strip()
LOCATION = "us-central1"

print(f"Active Lab Project ID: {PROJECT_ID}")
print(f"Target Infrastructure Location: {LOCATION}")

# Initialize the modern GenAI Client
# In a Google Cloud notebook environment, it handles authentication automatically
client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=LOCATION
)

Below, we recreate your original invocation function using the modern endpoint. We map your hyper-parameters (Max Tokens, Temperature, Top_p, and Top_k) into the standard configuration layout:

In [ ]:
def gemini_invoke(query):
    # Configure the generation parameters matching your original configuration layout
    config = types.GenerateContentConfig(
        candidate_count=1,
        max_output_tokens=1024,
        temperature=0.8,
        top_k=40,
        top_p=0.80,
    )

    # We use gemini-1.5-flash as the fast, lightweight model for testing
    response = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=f"Question: {query}",
        config=config
    )
    return f"Response from Model: {response.text}"


#### Test Queries

In [ ]:
query1 = 'What causes you to get ringworm?'
query2 = 'What is the monthly average of covid-19 patients in Virginia?'

In [ ]:
print(gemini_invoke(query1))

In [ ]:
print(gemini_invoke(query2))

### Gemini with REST API (Curl)

Utilizing the REST API is similar to Python, but uses raw HTTP curl commands. We specify the system parameters inside a valid JSON request payload.

In [ ]:
import json

# Define the query string
query = 'What is the monthly average of covid-19 patients in Virginia?'

# Build the payload - explicitly including "role": "user"
request_payload = {
    "contents": [
        {
            "role": "user", 
            "parts": [
                {
                    "text": f"Question: {query}"
                }
            ]
        }
    ],
    "generationConfig": {
        "temperature": 0.8,
        "maxOutputTokens": 1024,
        "topK": 40,
        "topP": 0.95
    }
}

# Write the JSON payload to a temporary file to prevent shell quoting issues
with open('request.json', 'w') as f:
    json.dump(request_payload, f)

Now we execute the bash curl pipeline. Note the updated URL route mapping to the modern gemini-2.5-flash model namespace:

In [ ]:
# Execute curl by reading from the request.json file (-d @request.json)
!curl -X POST \
    -H "Authorization: Bearer $(gcloud auth print-access-token)" \
    -H "Content-Type: application/json; charset=utf-8" \
    -d @request.json \
    "https://{LOCATION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/publishers/google/models/gemini-2.5-flash:streamGenerateContent"

### Clean up

In [ ]:
Please remember to delete or stop your Jupyter notebook runtime to prevent incurring ongoing sandbox charges!